<a href="https://colab.research.google.com/github/thiranesh/Thiranesh-codeboosters-2026/blob/main/day-7/Day_7_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers -q
print("Installation complete")
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
print("All libraries imported successfully")
print(f"ChromaDB version:{chromadb.__version__}")
client = chromadb.EphemeralClient()
collection = client.get_or_create_collection(name="my_collection")




Installation complete
All libraries imported successfully
ChromaDB version:1.5.9


In [ ]:
# Load the data and add it to the collection
df = pd.read_csv('/content/college_notes.csv')

# Ensure we have strings for the documents
documents = df['content'].tolist()
ids = [str(i) for i in range(len(documents))]
metadatas = df.drop(columns=['content']).to_dict(orient='records')

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)
print(f"Added {len(documents)} documents to the collection.")

Added 15 documents to the collection.


In [ ]:
all_documents = notes_df['content'].tolist()
all_ids = notes_df['note_id'].tolist()
all_metadatas = [{
    "subject": row['subject'],
    "topic": row['topic']
} for _, row in notes_df.iterrows()]
print(f"Documents prepared: {len(all_documents)}")
print(f"IDs prepared: {len(all_ids)}")
print(f"Metadata prepared: {len(all_metadatas)}")


Documents prepared: 15
IDs prepared: 15
Metadata prepared: 15


In [ ]:
print("\nLoading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded. Produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")



Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Produces vectors of size: 384 dimensions


/tmp/ipykernel_16163/3457278829.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")


In [ ]:
print("\nInitializing ChromaDB client...")
chroma_client = chromadb.Client()
collection_name = "college_notes_collection"

# Remove existing collection if it exists to ensure a clean start
try:
    chroma_client.delete_collection(name=collection_name)
    print(f"Existing collection '{collection_name}' deleted.")
except:
    pass # Collection might not exist, which is fine

collection = chroma_client.get_or_create_collection(collection_name)
print(f"ChromaDB collection '{collection_name}' created/retrieved.")
print(f"Documents in collection before adding: {collection.count()}")



Initializing ChromaDB client...
Existing collection 'college_notes_collection' deleted.
ChromaDB collection 'college_notes_collection' created/retrieved.
Documents in collection before adding: 0


In [ ]:
print("\nAdding all notes to ChromaDB collection...")
collection.add(
    documents=all_documents,
    ids=all_ids,
    metadatas=all_metadatas
)
print(f"Documents added to Collection. Total: {collection.count()}")



Adding all notes to ChromaDB collection...
Documents added to Collection. Total: 15


In [ ]:
query_text = "Tell me about data pipelines and transformations?"
query_embedding = model.encode(query_text).tolist()

print(f"\n--- Performing Semantic Search for: '{query_text}' ---")
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=['documents', 'distances', 'metadatas']
)

matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distances = results['distances'][0]
matched_meta = results['metadatas'][0]

for rank, (doc, doc_id, doc_dist, doc_meta) in enumerate(zip(matched_docs, matched_ids, matched_distances, matched_meta)):
    print(f"Rank: {rank} | ID: {doc_id} | Distance: {doc_dist:.4f}")
    print(f"Subject: {doc_meta['subject']} | Topic: {doc_meta['topic']}")
    print(f"Document: {doc[:150]}...") # Truncate for display
    print("-" * 30)



--- Performing Semantic Search for: 'Tell me about data pipelines and transformations?' ---
Rank: 0 | ID: N008 | Distance: 1.1357
Subject: Machine Learning | Topic: Feature Engineering
Document: Feature engineering is the process of selecting transforming and creating input variables that help machine learning models learn better. This include...
------------------------------
Rank: 1 | ID: N015 | Distance: 1.1359
Subject: Python Programming | Topic: Data Visualization
Document: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to c...
------------------------------
Rank: 2 | ID: N004 | Distance: 1.1948
Subject: Data Engineering | Topic: APIs and Data Collection
Document: An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data fr...
------------------------------


In [ ]:
query_1 = "How do I fix messy and incomplete data?"
results_1 = collection.query(
    query_texts=[query_1],
    n_results=3
)

# Note: Ensure 'show_results' is defined or use print(results_1)
print(results_1)

{'ids': [['2', '14', '13']], 'embeddings': None, 'documents': [['Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.', 'Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts line plots histograms and pie charts that help humans understand patterns in data.', 'Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table with rows and columns. Common operations include reading CSV files filtering rows grouping data and creating new columns.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'note_id': 'N003', 'topic': 'Data Cleaning', 'subject': 'Data Engineering'}, {'note_id': 'N015', 'subject': 'Pytho

In [ ]:

# 1. Load the dataset
notes_df = pd.read_csv("college_notes.csv")
print(f"\nDataset loaded. Shape: {notes_df.shape}, Columns: {list(notes_df.columns)}")

# 2. Prepare documents, IDs, and metadatas for ChromaDB
all_documents = notes_df['content'].tolist()
all_ids = notes_df['note_id'].tolist()
all_metadatas = [{
    "subject": row['subject'],
    "topic": row['topic']
} for _, row in notes_df.iterrows()]
print(f"Documents prepared: {len(all_documents)}")
print(f"IDs prepared: {len(all_ids)}")
print(f"Metadata prepared: {len(all_metadatas)}")

# 3. Load embedding model
print("\nLoading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded. Produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")

# 4. Initialize ChromaDB client and create/get a collection
print("\nInitializing ChromaDB client...")
chroma_client = chromadb.Client()
collection_name = "college_notes_collection"

# Remove existing collection if it exists to ensure a clean start
try:
    chroma_client.delete_collection(name=collection_name)
    print(f"Existing collection '{collection_name}' deleted.")
except:
    pass # Collection might not exist, which is fine

collection = chroma_client.get_or_create_collection(collection_name)
print(f"ChromaDB collection '{collection_name}' created/retrieved.")
print(f"Documents in collection before adding: {collection.count()}")

# 5. Add documents to ChromaDB
print("\nAdding all notes to ChromaDB collection...")
collection.add(
    documents=all_documents,
    ids=all_ids,
    metadatas=all_metadatas
)
print(f"Documents added to Collection. Total: {collection.count()}")

# 6. Perform a semantic search query
query_text = "Tell me about data pipelines and transformations?"
query_embedding = model.encode(query_text).tolist()

print(f"\n--- Performing Semantic Search for: '{query_text}' ---")
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=['documents', 'distances', 'metadatas']
)

matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distances = results['distances'][0]
matched_meta = results['metadatas'][0]

for rank, (doc, doc_id, doc_dist, doc_meta) in enumerate(zip(matched_docs, matched_ids, matched_distances, matched_meta)):
    print(f"Rank: {rank} | ID: {doc_id} | Distance: {doc_dist:.4f}")
    print(f"Subject: {doc_meta['subject']} | Topic: {doc_meta['topic']}")
    print(f"Document: {doc[:150]}...") # Truncate for display
    print("-" * 30)

# 7. Perform a filtered search query
filtered_query_text = "Machine learning models and evaluation."
filtered_query_embedding = model.encode(filtered_query_text).tolist()

print(f"\n--- Performing Filtered Semantic Search for: '{filtered_query_text}' ---")
print("Applying filter: {'subject': 'Machine Learning'}")
filtered_results = collection.query(
    query_embeddings=[filtered_query_embedding],
    n_results=2,
    where={"subject": "Machine Learning"},
    include=['documents', 'distances', 'metadatas']
)

filtered_matched_docs = filtered_results['documents'][0]
filtered_matched_ids = filtered_results['ids'][0]
filtered_matched_distances = filtered_results['distances'][0]
filtered_matched_meta = filtered_results['metadatas'][0]

for rank, (doc, doc_id, doc_dist, doc_meta) in enumerate(zip(filtered_matched_docs, filtered_matched_ids, filtered_matched_distances, filtered_matched_meta)):
    print(f"Rank: {rank} | ID: {doc_id} | Distance: {doc_dist:.4f}")
    print(f"Subject: {doc_meta['subject']} | Topic: {doc_meta['topic']}")
    print(f"Document: {doc[:150]}...") # Truncate for display
    print("-" * 30)

print("\n--- Workflow complete ---")


Dataset loaded. Shape: (15, 4), Columns: ['note_id', 'subject', 'topic', 'content']
Documents prepared: 15
IDs prepared: 15
Metadata prepared: 15

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_16163/3271990146.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")


Model loaded. Produces vectors of size: 384 dimensions

Initializing ChromaDB client...
ChromaDB collection 'college_notes_collection' created/retrieved.
Documents in collection before adding: 0

Adding all notes to ChromaDB collection...
Documents added to Collection. Total: 15

--- Performing Semantic Search for: 'Tell me about data pipelines and transformations?' ---
Rank: 0 | ID: N008 | Distance: 1.1357
Subject: Machine Learning | Topic: Feature Engineering
Document: Feature engineering is the process of selecting transforming and creating input variables that help machine learning models learn better. This include...
------------------------------
Rank: 1 | ID: N015 | Distance: 1.1359
Subject: Python Programming | Topic: Data Visualization
Document: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to c...
------------------------------
Rank: 2 | ID: N004 | Distance: 1.1948
Subject: Dat